# Notebook 02: Orchestration and passing data by location

This notebook builds a tiny local pipeline. The important idea is not the framework; it is the shape of the workflow.

You will pass file paths between steps instead of passing the full dataset object.

In [1]:
from pathlib import Path
import json
import pandas as pd

OUT = Path("artifacts_pipeline")
OUT.mkdir(exist_ok=True)

## 1. A tiny component example with Linda

This mirrors the basic idea of components and outputs, but with a changed example name.

In [2]:
def greet_customer(name: str) -> str:
    return f"Hello {name},"


def add_support_context(greeting_text: str, issue: str) -> str:
    return f"{greeting_text} I can help with your issue: {issue}"


greeting = greet_customer(name="Linda")
final_message = add_support_context(
    greeting_text=greeting,
    issue="the delivery address may need to be changed"
)

final_message

'Hello Linda, I can help with your issue: the delivery address may need to be changed'

## 2. Define pipeline components

Each component receives simple values or locations. It writes artifacts to disk and returns paths.

In [4]:
def create_raw_dataset(out_dir: Path) -> Path:
    rows = [
        {"ticket_id": "T-101", "message": "The product arrived with a missing part.", "reply": "Please share your order number and a photo of the missing part area so we can help."},
        {"ticket_id": "T-102", "message": "I need a copy of my invoice.", "reply": "Please share the email used for the order, and I’ll help locate the invoice."},
        {"ticket_id": "T-103", "message": "The setup instructions are confusing.", "reply": "I can help. Please tell me which step is unclear, and I’ll walk you through it."},
    ]
    path = out_dir / "raw_support_tickets.csv"
    pd.DataFrame(rows).to_csv(path, index=False)
    return path


def prepare_jsonl(raw_csv_uri: Path, out_dir: Path) -> Path:
    df = pd.read_csv(raw_csv_uri)
    instruction = "You are a support assistant. Draft a helpful reply."

    #Customer ticket: 
    prepared = pd.DataFrame({
            "input_text": instruction + df["message"],
            "output_text": df["reply"],
        })
    path = out_dir / "prepared_support_tickets.jsonl"
    prepared.to_json(path, orient="records", lines=True, force_ascii=False)
    return path


def validate_jsonl(jsonl_uri: Path) -> dict:
    errors = []
    rows = 0
    for line_number, line in enumerate(jsonl_uri.read_text(encoding="utf-8").splitlines(), start=1):
        rows += 1
        obj = json.loads(line)
        if "input_text" not in obj or "output_text" not in obj:
            errors.append(f"line {line_number}: missing input_text or output_text")
    return {"rows": rows, "errors": errors, "valid": len(errors) == 0}


def register_artifact(jsonl_uri: Path, validation_report: dict, out_dir: Path) -> Path:
    registry_record = {
        "artifact_type": "training_data",
        "uri": str(jsonl_uri),
        "validation": validation_report,
    }
    path = out_dir / "registry_record.json"
    path.write_text(json.dumps(registry_record, indent=2), encoding="utf-8")
    return path

## 3. Orchestrate the steps

The workflow is explicit. Step 2 depends on the output location from Step 1. Step 3 validates the artifact. Step 4 registers it.

In [5]:
raw_uri = create_raw_dataset(OUT)
jsonl_uri = prepare_jsonl(raw_csv_uri=raw_uri, out_dir=OUT)
validation = validate_jsonl(jsonl_uri=jsonl_uri)
registry_uri = register_artifact(jsonl_uri=jsonl_uri, validation_report=validation, out_dir=OUT)

print("raw_uri:", raw_uri)
print("jsonl_uri:", jsonl_uri)
print("validation:", validation)
print("registry_uri:", registry_uri)

raw_uri: artifacts_pipeline\raw_support_tickets.csv
jsonl_uri: artifacts_pipeline\prepared_support_tickets.jsonl
validation: {'rows': 3, 'errors': [], 'valid': True}
registry_uri: artifacts_pipeline\registry_record.json


## 4. Inspect the registry record

In [6]:
print(registry_uri.read_text(encoding="utf-8"))

{
  "artifact_type": "training_data",
  "uri": "artifacts_pipeline\\prepared_support_tickets.jsonl",
  "validation": {
    "rows": 3,
    "errors": [],
    "valid": true
  }
}


## Try it yourself

Add a new component called `check_minimum_rows(jsonl_uri, minimum_rows)` that fails if the dataset has fewer than 5 rows. Then add two more examples to `create_raw_dataset` until the pipeline passes.

The challenge is conceptual: where should the check live, and what should it return so the next step can decide whether to continue?